In [0]:
%run ./00_setup

In [0]:
validation_files = [
    "customers_validation.json",
    "orders_validation.json",
    "orders_fk_validation.json",
]

audit_rows = []

for file_name in validation_files:
    payload = load_json(f"{VALIDATION_PATH}/{file_name}")
    stats = payload.get("statistics", {})
    audit_rows.append((
        file_name.replace(".json", ""),
        bool(payload.get("success", False)),
        int(stats.get("evaluated_expectations", 0)),
        int(stats.get("successful_expectations", 0)),
        int(stats.get("unsuccessful_expectations", 0)),
        str(dt.datetime.utcnow())
    ))

audit_schema = T.StructType([
    T.StructField("validation_name", T.StringType(), False),
    T.StructField("success", T.BooleanType(), False),
    T.StructField("evaluated_expectations", T.IntegerType(), False),
    T.StructField("successful_expectations", T.IntegerType(), False),
    T.StructField("unsuccessful_expectations", T.IntegerType(), False),
    T.StructField("audit_ts_utc", T.StringType(), False),
])

audit_df = spark.createDataFrame(audit_rows, schema=audit_schema)

audit_df.write.mode("overwrite").format("delta").saveAsTable(f"workspace.silver.dq_validation_audit")

display(audit_df)

In [0]:
quarantine_df = spark.table(f"workspace.silver.orders_quarantine")

display(quarantine_df)

In [0]:
rule_frequency = (
    quarantine_df
    .select(F.explode("failed_rules").alias("failed_rule"))
    .groupBy("failed_rule")
    .count()
    .orderBy(F.desc("count"))
)

display(rule_frequency)